<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/model_experiment_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install mlflow kaggle

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
tracking_uri = setup_env()
print("MLflow →", tracking_uri)

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [2]:
from src.train_val_split import time_based_split
from src.features import (
    CalendarFeatures,
    LagFeatures,
    DropColumns,
    LAG_CONFIG_BASELINE,
    LAG_CONFIG_SUBMISSION,
)
from src.metrics import wmae
from src.config import DATA_DIR, TRACKING_URI

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from sklearn.pipeline import Pipeline

mlflow.set_experiment("lightGBM_Training")
print(f"MLflow tracking URI: {tracking_uri}")

MLflow tracking URI: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow


## split the data between train and validation

In [4]:
train_split, val_split = time_based_split(train, date_col="Date", val_weeks=39)
VAL_WEEKS = 39

print(f"train_split: {len(train_split):,} rows  {train_split['Date'].min().date()} -> {train_split['Date'].max().date()}")
print(f"val_split:   {len(val_split):,} rows  {val_split['Date'].min().date()} -> {val_split['Date'].max().date()}")

train_split: 305,982 rows  2010-02-05 -> 2012-01-27
val_split:   115,588 rows  2012-02-03 -> 2012-10-26


## Feature pipeline

`CalendarFeatures -> LagFeatures(**lag_config) -> DropColumns(["Date"])`.

`LagFeatures.fit` snapshots training history internally, so we `fit_transform` on
`train_split` (passing `y=Weekly_Sales` so the Pipeline forwards it to every step's
`fit`) and only `transform` on `val_split` — this is what keeps validation
leakage-free, since `val_split`'s own sales never enter `LagFeatures.history_`.


In [5]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import TransformedTargetRegressor



LGB_PARAMS = dict(
    objective="regression_l1",   # MAE objective pairs naturally with a WMAE eval metric
    metric="mae",               # Changed from "None" to "mae" to ensure weights are preserved for feval
    boosting_type="gbdt",
    num_leaves=63,
    max_depth=-1,
    learning_rate=0.05,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    min_data_in_leaf=20,
    verbosity=-1,
    seed=42,
)

NUM_BOOST_ROUND = 3000
EARLY_STOPPING_ROUNDS = 100

# log1p that also works for the negative (returns) rows
def signed_log1p(y):
    return np.sign(y) * np.log1p(np.abs(y))

def signed_expm1(z):
    return np.sign(z) * np.expm1(np.abs(z))

CATEGORICAL_COLS = ["Store", "Dept"]

def holiday_weights(is_holiday):
    return np.where(np.asarray(is_holiday) == 1, 5.0, 1.0)


def to_lgb_frame(df, train_categories=None):
    """Split a transformed frame into X (with Store/Dept as category dtype), y, weights."""
    y = df["Weekly_Sales"].to_numpy(dtype=float) if "Weekly_Sales" in df.columns else None
    X = df.drop(columns=["Weekly_Sales"]).copy() if "Weekly_Sales" in df.columns else df.copy()
    w = holiday_weights(df["IsHoliday"])

    for col in CATEGORICAL_COLS:
        if train_categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=train_categories[col])

    return X, y, w


class ToLGBFrame(BaseEstimator):
    """sklearn-compatible wrapper around to_lgb_frame, so categorical
    alignment (fitted on train) can live inside a Pipeline step."""
    def fit(self, X, y=None):
        X_lgb, _, _ = to_lgb_frame(X)
        self.train_categories_ = {c: X_lgb[c].cat.categories for c in CATEGORICAL_COLS}
        return self

    def transform(self, X):
        X_lgb, _, _ = to_lgb_frame(X, train_categories=self.train_categories_)
        return X_lgb


class LGBMBoosterRegressor(BaseEstimator, RegressorMixin):
    """sklearn-compatible wrapper around a raw lgb.Booster."""
    def __init__(self, num_boost_round=100, **lgb_params):
        self.num_boost_round = num_boost_round
        self.lgb_params = lgb_params

    def fit(self, X, y, sample_weight=None):
        train_set = lgb.Dataset(
            X, label=y, weight=sample_weight,
            categorical_feature=CATEGORICAL_COLS, free_raw_data=False,
        )
        self.booster_ = lgb.train(self.lgb_params, train_set, num_boost_round=self.num_boost_round)
        return self

    def predict(self, X):
        return self.booster_.predict(X, num_iteration=self.booster_.best_iteration)


def build_full_pipeline(lag_config, log_target, num_boost_round, **lgb_overrides):
    """The Pipeline the assignment wants: raw df in -> predictions out."""
    regressor = LGBMBoosterRegressor(num_boost_round=num_boost_round, **{**LGB_PARAMS, **lgb_overrides})
    if log_target:
        regressor = TransformedTargetRegressor(regressor=regressor, func=signed_log1p, inverse_func=signed_expm1)
    return Pipeline([
        ("calendar", CalendarFeatures()),
        ("lags",     LagFeatures(**lag_config)),
        ("drop",     DropColumns(columns=["Date"])),
        ("to_lgb",   ToLGBFrame()),
        ("model",    regressor),
    ])

## Train + log one config to MLflow

Runs the full fit -> WMAE-early-stopped train -> log cycle for a given lag config,
so we can call it once for `LAG_CONFIG_BASELINE` and once for `LAG_CONFIG_SUBMISSION`
and compare both as separate MLflow runs.


In [6]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
import mlflow.sklearn
import mlflow.lightgbm
from sklearn.compose import TransformedTargetRegressor
import mlflow # Import mlflow for active_run()

# Redefine wmae here to ensure correct signature and implementation
def wmae(preds, lgb_train):
    """
    Weighted Mean Absolute Error for LightGBM.
    Ensures correct signature for LightGBM's feval.
    """
    labels = lgb_train.get_label()
    weights = lgb_train.get_weight()
    # Handle case where get_weight() returns None (e.g., when all weights are 1)
    if weights is None:
        weights = np.ones_like(labels)
    return 'wmae', (np.sum(weights * np.abs(labels - preds)) / np.sum(weights)), False

# Helper function to calculate WMAE on a given scale
def calculate_wmae(y_true, y_pred, weights):
    """
    Calculates WMAE given true values, predictions, and weights.
    """
    if weights is None:
        weights = np.ones_like(y_true)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


# Dictionary to store results for comparison
results = {}

def build_feature_pipeline(lag_config):
    return Pipeline([
        ("calendar", CalendarFeatures()),
        ("lags",     LagFeatures(**lag_config)),
        ("drop",     DropColumns(columns=["Date"])), # Explicitly passing columns here
    ])


def train_and_log(run_name,  lag_config, log_target=False, weighted=False, **lgb_overrides):
      # Build the feature engineering pipeline
      feature_pipeline = build_feature_pipeline(lag_config)

      # Capture original validation target for later comparison
      # `val_split` contains the original 'Weekly_Sales' values before any transformations.
      y_val_original_untransformed = val_split["Weekly_Sales"].copy()


      # Fit the feature pipeline on the training data
      feature_pipeline.fit(train_split, y=train_split["Weekly_Sales"])

      # Transform training and validation data
      train_t = feature_pipeline.transform(train_split)
      val_t = feature_pipeline.transform(val_split)


      # Apply target transformation if log_target is True
      if log_target:
          train_t["Weekly_Sales"] = signed_log1p(train_t["Weekly_Sales"])
          val_t["Weekly_Sales"] = signed_log1p(val_t["Weekly_Sales"])

      X_train, y_train, w_train_computed = to_lgb_frame(train_t)
      train_categories = {c: X_train[c].cat.categories for c in CATEGORICAL_COLS}
      # y_val_transformed will hold the potentially log-transformed target for validation set
      X_val, y_val_transformed, w_val_computed = to_lgb_frame(val_t, train_categories=train_categories)

      # Conditionally apply weights
      # `weighted` is the ablation knob: it only decides whether the TRAINING
      # weights are the 5x holiday weights or flat ones. Evaluation is ALWAYS
      # holiday-weighted, otherwise the unweighted runs report plain MAE and
      # can't be compared with the weighted ones (or with the other models).
      actual_w_train = w_train_computed if weighted else np.ones_like(y_train)

      train_set = lgb.Dataset(
          X_train, label=y_train, weight=actual_w_train,
          categorical_feature=CATEGORICAL_COLS, free_raw_data=False,
      )
      val_set = lgb.Dataset(
          X_val, label=y_val_transformed, weight=w_val_computed, reference=train_set,
          categorical_feature=CATEGORICAL_COLS, free_raw_data=False,
      )
          # Log preprocessing related info
          # Log the fitted feature engineering pipeline
          # mlflow.sklearn.log_model(pipeline, artifact_path="feature_pipeline", serialization_format="cloudpickle"
      # Nested run for Training
      with mlflow.start_run(run_name=run_name) as run:
          mlflow.log_params({**LGB_PARAMS, **lgb_overrides}) # Log LightGBM hyperparameters

          # num_boost_round is an lgb.train() argument, not a params-dict key, so
          # pull it out of the overrides before merging the rest into the params.
          params = {**LGB_PARAMS,
                    **{k: v for k, v in lgb_overrides.items() if k != "num_boost_round"}}
          n_rounds = lgb_overrides.get("num_boost_round", NUM_BOOST_ROUND)

          booster = lgb.train(
              params,   # LGB_PARAMS plus whatever this run overrides
              train_set,
              num_boost_round=n_rounds,
              valid_sets=[train_set, val_set],
              valid_names=["train", "val"],
              feval=wmae,
              callbacks=[
                  lgb.early_stopping(EARLY_STOPPING_ROUNDS),
                  lgb.log_evaluation(100),
              ],
          )

          mlflow.log_params({"lags": str(lag_config["lags"]), "windows": str(lag_config["windows"]),
                           "log_target": log_target, "weighted": weighted, "val_weeks": VAL_WEEKS})

          # Log WMAE on the scale it was trained (transformed if log_target=True)
          train_wmae_transformed_scale = booster.best_score["train"]["wmae"]
          mlflow.log_metric("train_wmae_transformed_scale", train_wmae_transformed_scale)

          val_wmae_transformed_scale = booster.best_score["val"]["wmae"]
          mlflow.log_metric("val_wmae_transformed_scale", val_wmae_transformed_scale)
          mlflow.log_metric("best_iteration", booster.best_iteration)


          # Calculate WMAE on the original scale for fair comparison
          train_preds = booster.predict(X_train, num_iteration=booster.best_iteration)
          if log_target:
              train_preds_original_scale = signed_expm1(train_preds)
              y_train_original_untransformed = signed_expm1(y_train) # Need to inverse transform y_train for original scale comparison
          else:
              train_preds_original_scale = train_preds
              y_train_original_untransformed = y_train

          train_wmae_original_scale = calculate_wmae(y_train_original_untransformed, train_preds_original_scale, w_train_computed)
          mlflow.log_metric("train_wmae_original_scale", train_wmae_original_scale)


          val_preds = booster.predict(X_val, num_iteration=booster.best_iteration)
          if log_target:
              val_preds_original_scale = signed_expm1(val_preds)
          else:
              val_preds_original_scale = val_preds

          # always the holiday weights, never flat ones -- this is the number the
          # scoreboard sorts on and the number we compare across models
          val_wmae_original_scale = calculate_wmae(y_val_original_untransformed, val_preds_original_scale, w_val_computed)
          mlflow.log_metric("val_wmae_original_scale", val_wmae_original_scale)


          ax = lgb.plot_importance(booster, max_num_features=20, importance_type="gain")
          fig = ax.figure
          fig.tight_layout()
          fig_path = f"feature_importance_{run_name}.png"
          fig.savefig(fig_path, bbox_inches="tight")
          plt.close(fig)
          mlflow.log_artifact(fig_path)

          # Update results dictionary to store the WMAE on the original scale
          results[run_name] = (val_wmae_original_scale, dict(lag_cfg=lag_config, log_target=log_target,
                                         weighted=weighted, overrides=lgb_overrides))

          # Log the trained LightGBM model
          # mlflow.lightgbm.log_model(booster, artifact_path="lgbm_model") # Changed to log booster, not pipeline

          print(f"[{run_name}] best_iteration={booster.best_iteration}  train_wmae_original_scale={train_wmae_original_scale:.2f}  val_wmae_original_scale={val_wmae_original_scale:.2f}")

      return booster, val_wmae_original_scale, feature_pipeline, run.info.run_id # Return the original scale WMAE

 ## Train


In [7]:
print("\n--- Running for lag_submission_log1p ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_log1p", lag_config=LAG_CONFIG_SUBMISSION,
                                                             log_target = True)



--- Running for lag_submission_log1p ---


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.339902	train's wmae: 0.339902	val's l1: 0.295628	val's wmae: 0.295628
[200]	train's l1: 0.290473	train's wmae: 0.290473	val's l1: 0.277814	val's wmae: 0.277814
[300]	train's l1: 0.263126	train's wmae: 0.263126	val's l1: 0.271438	val's wmae: 0.271438
[400]	train's l1: 0.245761	train's wmae: 0.245761	val's l1: 0.268003	val's wmae: 0.268003
[500]	train's l1: 0.234192	train's wmae: 0.234192	val's l1: 0.266341	val's wmae: 0.266341
[600]	train's l1: 0.224073	train's wmae: 0.224073	val's l1: 0.264964	val's wmae: 0.264964
[700]	train's l1: 0.21752	train's wmae: 0.21752	val's l1: 0.264046	val's wmae: 0.264046
[800]	train's l1: 0.21108	train's wmae: 0.21108	val's l1: 0.26348	val's wmae: 0.26348
[900]	train's l1: 0.20653	train's wmae: 0.20653	val's l1: 0.263177	val's wmae: 0.263177
[1000]	train's l1: 0.202858	train's wmae: 0.202858	val's l1: 0.262918	val's wmae: 0.262918
[1100]	train's l1: 0.199674	train's wmae: 0.1

In [8]:
mlflow.set_tracking_uri(TRACKING_URI)



# Train and log for LAG_CONFIG_BASELINE
print("\n--- Running for lag_baseline ---")
_, val_wmae_baseline, _, run_id_baseline = train_and_log(run_name="lag_baseline", lag_config=LAG_CONFIG_BASELINE)

# Train and log for LAG_CONFIG_SUBMISSION
print("\n--- Running for lag_submission ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission", lag_config=LAG_CONFIG_SUBMISSION)

# print("\n--- Running for lag_submission_log1p ---")
# _, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_log1p", lag_config=LAG_CONFIG_SUBMISSION,
#                                                              log_target = True)

print("\n--- Running for lag_submission_weighted ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_weighted", lag_config=LAG_CONFIG_SUBMISSION,
                                                             weighted = True)

print("\n--- Running for lag_submission_log1p_weighted ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_log1p_weighted", lag_config=LAG_CONFIG_SUBMISSION,
                                                             log_target = True, weighted = True)



# print("\n--- Comparing Results ---")
# print(f"LAG_CONFIG_BASELINE WMAE: {val_wmae_baseline:.2f}")
# print(f"LAG_CONFIG_SUBMISSION WMAE: {val_wmae_submission:.2f}")

# # Determine the best run based on WMAE (lower is better)
# if val_wmae_baseline <= val_wmae_submission:
#     best_config_name = "LAG_CONFIG_BASELINE"
#     best_run_id = run_id_baseline
# else:
#     best_config_name = "LAG_CONFIG_SUBMISSION"
#     best_run_id = run_id_submission

# print(f"\nThe best performing configuration is: {best_config_name} (Run ID: {best_run_id})")


--- Running for lag_baseline ---


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 1736.74	train's wmae: 1736.74	val's l1: 7249.09	val's wmae: 7249.09
[200]	train's l1: 1522.94	train's wmae: 1522.94	val's l1: 6319.84	val's wmae: 6319.84
[300]	train's l1: 1436.69	train's wmae: 1436.69	val's l1: 6291.03	val's wmae: 6291.03
[400]	train's l1: 1380.97	train's wmae: 1380.97	val's l1: 6251.51	val's wmae: 6251.51
[500]	train's l1: 1339.1	train's wmae: 1339.1	val's l1: 6248.26	val's wmae: 6248.26
[600]	train's l1: 1307.13	train's wmae: 1307.13	val's l1: 6135.57	val's wmae: 6135.57
[700]	train's l1: 1284.76	train's wmae: 1284.76	val's l1: 6064.37	val's wmae: 6064.37
[800]	train's l1: 1263.59	train's wmae: 1263.59	val's l1: 5998.96	val's wmae: 5998.96
[900]	train's l1: 1244.51	train's wmae: 1244.51	val's l1: 5988.78	val's wmae: 5988.78
Early stopping, best iteration is:
[876]	train's l1: 1250.18	train's wmae: 1250.18	val's l1: 5960.66	val's wmae: 5960.66
[lag_baseline] best_iteration=876  train_wmae

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 2767.01	train's wmae: 2767.01	val's l1: 1854.12	val's wmae: 1854.12
[200]	train's l1: 2408.87	train's wmae: 2408.87	val's l1: 1735.19	val's wmae: 1735.19
[300]	train's l1: 2193.77	train's wmae: 2193.77	val's l1: 1713.05	val's wmae: 1713.05
[400]	train's l1: 2025.04	train's wmae: 2025.04	val's l1: 1701.91	val's wmae: 1701.91
[500]	train's l1: 1912.98	train's wmae: 1912.98	val's l1: 1696.76	val's wmae: 1696.76
[600]	train's l1: 1810.77	train's wmae: 1810.77	val's l1: 1692.7	val's wmae: 1692.7
[700]	train's l1: 1734.51	train's wmae: 1734.51	val's l1: 1690.83	val's wmae: 1690.83
[800]	train's l1: 1665.37	train's wmae: 1665.37	val's l1: 1688.66	val's wmae: 1688.66
[900]	train's l1: 1619.44	train's wmae: 1619.44	val's l1: 1683.66	val's wmae: 1683.66
[1000]	train's l1: 1574.63	train's wmae: 1574.63	val's l1: 1683.67	val's wmae: 1683.67
Early stopping, best iteration is:
[958]	train's l1: 1592.28	train's wmae: 1592

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 2974.96	train's wmae: 2974.96	val's l1: 1826.47	val's wmae: 1826.47
[200]	train's l1: 2611.62	train's wmae: 2611.62	val's l1: 1739.8	val's wmae: 1739.8
[300]	train's l1: 2374.74	train's wmae: 2374.74	val's l1: 1720.61	val's wmae: 1720.61
[400]	train's l1: 2184.05	train's wmae: 2184.05	val's l1: 1712.14	val's wmae: 1712.14
[500]	train's l1: 2065.37	train's wmae: 2065.37	val's l1: 1708.94	val's wmae: 1708.94
[600]	train's l1: 1968.91	train's wmae: 1968.91	val's l1: 1706.39	val's wmae: 1706.39
[700]	train's l1: 1895.05	train's wmae: 1895.05	val's l1: 1704.74	val's wmae: 1704.74
[800]	train's l1: 1833.73	train's wmae: 1833.73	val's l1: 1702.1	val's wmae: 1702.1
[900]	train's l1: 1788.74	train's wmae: 1788.74	val's l1: 1701.07	val's wmae: 1701.07
[1000]	train's l1: 1747.53	train's wmae: 1747.53	val's l1: 1700.98	val's wmae: 1700.98
Early stopping, best iteration is:
[950]	train's l1: 1767.31	train's wmae: 1767.3

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.342909	train's wmae: 0.342909	val's l1: 0.296374	val's wmae: 0.296374
[200]	train's l1: 0.295044	train's wmae: 0.295044	val's l1: 0.281999	val's wmae: 0.281999
[300]	train's l1: 0.269994	train's wmae: 0.269994	val's l1: 0.276517	val's wmae: 0.276517
[400]	train's l1: 0.251867	train's wmae: 0.251867	val's l1: 0.272195	val's wmae: 0.272195
[500]	train's l1: 0.23985	train's wmae: 0.23985	val's l1: 0.270151	val's wmae: 0.270151
[600]	train's l1: 0.229624	train's wmae: 0.229624	val's l1: 0.268759	val's wmae: 0.268759
[700]	train's l1: 0.223067	train's wmae: 0.223067	val's l1: 0.268051	val's wmae: 0.268051
[800]	train's l1: 0.216812	train's wmae: 0.216812	val's l1: 0.267328	val's wmae: 0.267328
[900]	train's l1: 0.21287	train's wmae: 0.21287	val's l1: 0.267133	val's wmae: 0.267133
[1000]	train's l1: 0.209006	train's wmae: 0.209006	val's l1: 0.266903	val's wmae: 0.266903
[1100]	train's l1: 0.20646	train's wmae: 

In [9]:
# pick the best target + weighting combo from the four base runs, then tune around it
base = ["lag_submission", "lag_submission_log1p",
        "lag_submission_weighted", "lag_submission_log1p_weighted"]
best_base = min(base, key=lambda k: results[k][0])
use_log    = results[best_base][1]["log_target"]
use_weight = results[best_base][1]["weighted"]
print(f"tuning around {best_base}: log_target={use_log}, weighted={use_weight}")

LGB_grid = [
   dict(
    num_leaves=31,
    max_depth=6,
    learning_rate=0.05
),
   dict(
    num_leaves=127,
    max_depth=-1,
    learning_rate=0.05
),
   dict(
    num_leaves=127,
    learning_rate=0.02,
    num_boost_round=3000
),
   dict(
    num_leaves=63,
    min_data_in_leaf=100,
    lambda_l2=5
),
   dict(
    num_leaves=63,
    feature_fraction=0.7,
    bagging_fraction=0.7,
    bagging_freq=5
)
]
for g in LGB_grid:
    name = "tuned_" + "_".join(f"{k}{v}" for k, v in g.items())
    _, val_wmae_submission, _, run_id_submission = train_and_log(
        run_name=name,
        lag_config=LAG_CONFIG_SUBMISSION,
        log_target=use_log,
        weighted=use_weight,
        **g,
    )

tuning around lag_submission: log_target=False, weighted=False


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 3421.24	train's wmae: 3421.24	val's l1: 1957.58	val's wmae: 1957.58
[200]	train's l1: 3000.04	train's wmae: 3000.04	val's l1: 1821.12	val's wmae: 1821.12
[300]	train's l1: 2751.2	train's wmae: 2751.2	val's l1: 1776.81	val's wmae: 1776.81
[400]	train's l1: 2563.05	train's wmae: 2563.05	val's l1: 1745.92	val's wmae: 1745.92
[500]	train's l1: 2437.36	train's wmae: 2437.36	val's l1: 1733.32	val's wmae: 1733.32
[600]	train's l1: 2328	train's wmae: 2328	val's l1: 1726.07	val's wmae: 1726.07
[700]	train's l1: 2236.19	train's wmae: 2236.19	val's l1: 1715.27	val's wmae: 1715.27
[800]	train's l1: 2156.38	train's wmae: 2156.38	val's l1: 1708.67	val's wmae: 1708.67
[900]	train's l1: 2096.75	train's wmae: 2096.75	val's l1: 1705.47	val's wmae: 1705.47
[1000]	train's l1: 2026.68	train's wmae: 2026.68	val's l1: 1700.71	val's wmae: 1700.71
[1100]	train's l1: 1972.56	train's wmae: 1972.56	val's l1: 1698.49	val's wmae: 1698.4

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 2451.29	train's wmae: 2451.29	val's l1: 1785.63	val's wmae: 1785.63
[200]	train's l1: 2077.42	train's wmae: 2077.42	val's l1: 1694.96	val's wmae: 1694.96
[300]	train's l1: 1848.93	train's wmae: 1848.93	val's l1: 1684.93	val's wmae: 1684.93
[400]	train's l1: 1700.17	train's wmae: 1700.17	val's l1: 1679.56	val's wmae: 1679.56
[500]	train's l1: 1607.06	train's wmae: 1607.06	val's l1: 1677.14	val's wmae: 1677.14
[600]	train's l1: 1533.06	train's wmae: 1533.06	val's l1: 1676.36	val's wmae: 1676.36
[700]	train's l1: 1473.16	train's wmae: 1473.16	val's l1: 1675.41	val's wmae: 1675.41
[800]	train's l1: 1430.17	train's wmae: 1430.17	val's l1: 1674.24	val's wmae: 1674.24
[900]	train's l1: 1393.53	train's wmae: 1393.53	val's l1: 1675.68	val's wmae: 1675.68
Early stopping, best iteration is:
[803]	train's l1: 1429.64	train's wmae: 1429.64	val's l1: 1674.17	val's wmae: 1674.17
[tuned_num_leaves127_max_depth-1_learning_r

...

Training until validation scores don't improve for 100 rounds
[100]	train's l1: 4391.65	train's wmae: 4391.65	val's l1: 3574.81	val's wmae: 3574.81
[200]	train's l1: 2795.56	train's wmae: 2795.56	val's l1: 1937.94	val's wmae: 1937.94
[300]	train's l1: 2365.22	train's wmae: 2365.22	val's l1: 1718.99	val's wmae: 1718.99
[400]	train's l1: 2189.41	train's wmae: 2189.41	val's l1: 1688.43	val's wmae: 1688.43
[500]	train's l1: 2065.5	train's wmae: 2065.5	val's l1: 1678.03	val's wmae: 1678.03
[600]	train's l1: 1950.25	train's wmae: 1950.25	val's l1: 1673.36	val's wmae: 1673.36
[700]	train's l1: 1865.41	train's wmae: 1865.41	val's l1: 1669.8	val's wmae: 1669.8
[800]	train's l1: 1783.92	train's wmae: 1783.92	val's l1: 1668.63	val's wmae: 1668.63
[900]	train's l1: 1727.55	train's wmae: 1727.55	val's l1: 1666.3	val's wmae: 1666.3
[1000]	train's l1: 1675.16	train's wmae: 1675.16	val's l1: 1666.38	val's wmae: 1666.38
Early stopping, best iteration is:
[934]	train's l1: 1709.42	train's wmae: 1709.42	

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 2786.1	train's wmae: 2786.1	val's l1: 1837.06	val's wmae: 1837.06
[200]	train's l1: 2427.32	train's wmae: 2427.32	val's l1: 1722.83	val's wmae: 1722.83
[300]	train's l1: 2194.32	train's wmae: 2194.32	val's l1: 1704.32	val's wmae: 1704.32
[400]	train's l1: 2036.32	train's wmae: 2036.32	val's l1: 1694.98	val's wmae: 1694.98
[500]	train's l1: 1927.41	train's wmae: 1927.41	val's l1: 1690.91	val's wmae: 1690.91
[600]	train's l1: 1826.47	train's wmae: 1826.47	val's l1: 1689.63	val's wmae: 1689.63
[700]	train's l1: 1754.22	train's wmae: 1754.22	val's l1: 1685.58	val's wmae: 1685.58
[800]	train's l1: 1688.37	train's wmae: 1688.37	val's l1: 1682.55	val's wmae: 1682.55
[900]	train's l1: 1639.84	train's wmae: 1639.84	val's l1: 1681.88	val's wmae: 1681.88
[1000]	train's l1: 1599.21	train's wmae: 1599.21	val's l1: 1680.84	val's wmae: 1680.84
[1100]	train's l1: 1567.66	train's wmae: 1567.66	val's l1: 1680.85	val's wmae: 

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 2832.89	train's wmae: 2832.89	val's l1: 1861.28	val's wmae: 1861.28
[200]	train's l1: 2458.82	train's wmae: 2458.82	val's l1: 1739.74	val's wmae: 1739.74
[300]	train's l1: 2260.98	train's wmae: 2260.98	val's l1: 1715.15	val's wmae: 1715.15
[400]	train's l1: 2071.7	train's wmae: 2071.7	val's l1: 1703.74	val's wmae: 1703.74
[500]	train's l1: 1957.93	train's wmae: 1957.93	val's l1: 1693.2	val's wmae: 1693.2
[600]	train's l1: 1862.26	train's wmae: 1862.26	val's l1: 1686.35	val's wmae: 1686.35
[700]	train's l1: 1781.99	train's wmae: 1781.99	val's l1: 1683.85	val's wmae: 1683.85
[800]	train's l1: 1718.77	train's wmae: 1718.77	val's l1: 1678.21	val's wmae: 1678.21
[900]	train's l1: 1670.9	train's wmae: 1670.9	val's l1: 1676.27	val's wmae: 1676.27
[1000]	train's l1: 1627.99	train's wmae: 1627.99	val's l1: 1674.76	val's wmae: 1674.76
Early stopping, best iteration is:
[971]	train's l1: 1641.33	train's wmae: 1641.33	

In [10]:

# all runs side by side
board = pd.Series({k: v[0] for k, v in results.items()}).sort_values()
print(board.round(1).to_string())


tuned_num_leaves127_learning_rate0.02_num_boost_round3000                   1665.6
tuned_num_leaves63_feature_fraction0.7_bagging_fraction0.7_bagging_freq5    1673.7
tuned_num_leaves127_max_depth-1_learning_rate0.05                           1674.2
tuned_num_leaves63_min_data_in_leaf100_lambda_l25                           1680.5
lag_submission                                                              1682.5
tuned_num_leaves31_max_depth6_learning_rate0.05                             1688.4
lag_submission_log1p                                                        1691.4
lag_submission_weighted                                                     1700.2
lag_submission_log1p_weighted                                               1766.1
lag_baseline                                                                5960.7


 ## Log the best model

In [11]:
best_name = board.index[0]
_, spec = results[best_name]
client = mlflow.tracking.MlflowClient()
runs = client.search_runs(
    experiment_ids=[mlflow.get_experiment_by_name("lightGBM_Training").experiment_id],
    filter_string=f"tags.mlflow.runName = '{best_name}'"
)
best_iteration = int(runs[0].data.metrics["best_iteration"])
print("refitting on full train:", best_name, spec)

# num_boost_round may also live in the tuned overrides; best_iteration (found by
# early stopping) wins, so drop it from the overrides to avoid a duplicate kwarg
overrides = {k: v for k, v in spec["overrides"].items() if k != "num_boost_round"}
final_pipe = build_full_pipeline(
    spec["lag_cfg"], spec["log_target"],
    num_boost_round=best_iteration, **overrides,
)

fit_kwargs = {}
if spec["weighted"]:
    fit_kwargs["model__sample_weight"] = holiday_weights(train["IsHoliday"])
    if spec["log_target"]:
        fit_kwargs = {f"model__regressor__sample_weight": fit_kwargs.pop("model__sample_weight")}

final_pipe.fit(train.drop(columns=["Weekly_Sales"]), train["Weekly_Sales"], **fit_kwargs)

preds = final_pipe.predict(test)   # raw, unpreprocessed test frame straight through
assert np.isfinite(preds).all()

with mlflow.start_run(run_name=f"final_full_train__{best_name}"):
    mlflow.log_params({"val_wmae_of_chosen_cfg": results[best_name][0],
                        "chosen_run": best_name, "refit_on": "full_train",
                        "weighted": spec["weighted"], "best_iteration": best_iteration})
    mlflow.sklearn.log_model(final_pipe, "model", serialization_format="cloudpickle")

print("final pipeline logged | sample test preds:", preds[:3].round(1))

refitting on full train: tuned_num_leaves127_learning_rate0.02_num_boost_round3000 {'lag_cfg': {'lags': (52, 53, 104), 'windows': ()}, 'log_target': False, 'weighted': False, 'overrides': {'num_leaves': 127, 'learning_rate': 0.02, 'num_boost_round': 3000}}


2026/07/10 10:56:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 10:56:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_full_train__tuned_num_leaves127_learning_rate0.02_num_boost_round3000 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/5/runs/133c5e00a7e24e7fbdc809efc840e580
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/5
final pipeline logged | sample test preds: [36671.5 22043.7 20795.5]
